# Agente inversor: resultados en el período held-out

Evaluación final del agente congelado en la parte 12 (DQN, recompensa v1,
3 semillas) sobre el período de **test, 2016-01-04 → 2018-08-29**, que no se
usó para ninguna decisión. Se ejecuta una sola vez, con el procedimiento
fijado en las partes 12 y 13: `evaluation.compare` verifica los hashes de los
modelos antes de evaluarlos.

23. Curva de capital del agente frente a los baselines
24–26. Retorno acumulado, máximo drawdown y costos
27. Evolución de las posiciones del agente
28. Comparación entre las versiones de la recompensa
29. Comportamientos inesperados del agente

In [ ]:
import os
from pathlib import Path

# Ejecutar desde la raíz del repositorio, como la CLI.
if Path.cwd().name == "notebooks":
    os.chdir("..")

import matplotlib.pyplot as plt
import pandas as pd
from stable_baselines3 import DQN

from rl_games_sb import backtest, evaluation, investor

PERIOD = "test"

runs = evaluation.compare(PERIOD)  # verifica los hashes del agente congelado
metrics = evaluation.per_run(runs)
table = evaluation.summary(metrics)

agent_days = dict(enumerate(runs[evaluation.AGENT]))  # semilla -> filas diarias
bh_days = runs[evaluation.BUY_AND_HOLD][0]
prices = investor.load()["close"].reindex(bh_days.index)
asset_return = prices.pct_change().shift(-1)  # retorno del día t al t+1, el que gana la decisión de t
print(f"{PERIOD}: {bh_days.index[0].date()} → {bh_days.index[-1].date()}, {len(bh_days) - 1} decisiones")

## 23. Curva de capital del agente frente a los baselines

In [ ]:
evaluation.plot_equity(runs, f"Valor de $100.000 en {PERIOD}: agente RL (3 semillas) frente a los baselines")

**Interpretación:** (se completa con los resultados)

## 24–26. Retorno acumulado, máximo drawdown y costos

In [ ]:
report = pd.DataFrame({
    "Retorno acumulado": table["total_return"],
    "Máximo drawdown": table["max_drawdown"],
    "Costos": table["costs"],
    "Costos (% del capital inicial)": table["costs"] / investor.INITIAL_CAPITAL,
}).rename_axis("Estrategia")
report.style.format({
    "Retorno acumulado": "{:+.1%}",
    "Máximo drawdown": "{:+.1%}",
    "Costos": "${:,.0f}",
    "Costos (% del capital inicial)": "{:.2%}",
})

In [ ]:
LABELS = {
    "total_return": "retorno acumulado", "annual_return": "retorno anual", "volatility": "volatilidad anual",
    "sharpe": "Sharpe", "max_drawdown": "máximo drawdown", "trades": "operaciones", "costs": "costos ($)",
    "exposure": "exposición media",
}
FORMATS = {
    "retorno acumulado": "{:+.1%}", "retorno anual": "{:+.1%}", "volatilidad anual": "{:.1%}",
    "Sharpe": "{:.2f}", "máximo drawdown": "{:+.1%}", "operaciones": "{:.0f}", "costos ($)": "{:,.0f}",
    "exposición media": "{:.2f}",
}
random_metrics = metrics.loc[evaluation.RANDOM]
detail = pd.concat([
    metrics.loc[evaluation.AGENT].rename(index=lambda i: f"{evaluation.AGENT}, semilla {i}"),
    table.loc[[evaluation.AGENT]].rename(index={evaluation.AGENT: f"{evaluation.AGENT}, media"}),
    metrics.loc[evaluation.BUY_AND_HOLD].rename(index=lambda i: evaluation.BUY_AND_HOLD),
    random_metrics.quantile([0.05, 0.5, 0.95]).rename(index=lambda q: f"{evaluation.RANDOM}, percentil {q:.0%}"),
]).drop(columns="final_value").rename(columns=LABELS)
detail.style.format(FORMATS)

In [ ]:
# ¿En qué percentil del agente aleatorio queda cada semilla? (mayor es mejor en las tres métricas)
rank = pd.DataFrame({
    seed: {
        "retorno acumulado": (random_metrics["total_return"] < m["total_return"]).mean(),
        "máximo drawdown": (random_metrics["max_drawdown"] < m["max_drawdown"]).mean(),
        "Sharpe": (random_metrics["sharpe"] < m["sharpe"]).mean(),
    }
    for seed, m in metrics.loc[evaluation.AGENT].iterrows()
}).T.rename_axis("semilla")
print("Fracción de los 200 agentes aleatorios que cada semilla supera:")
rank.style.format("{:.0%}")

**Interpretación:** (se completa con los resultados)

## 27. Evolución de las posiciones del agente

In [ ]:
fig, axes = plt.subplots(len(agent_days) + 1, 1, figsize=(13, 8), sharex=True,
                         height_ratios=[2] + [1] * len(agent_days))
axes[0].plot(prices, color="black", lw=1)
axes[0].set_ylabel("precio GOOGL")
axes[0].set_title(f"Posición diaria de cada semilla en {PERIOD}")
for ax, (seed, days) in zip(axes[1:], agent_days.items()):
    ax.step(days.index, days["position"], where="post", color="tab:blue")
    ax.set_yticks([0, 0.5, 1], ["0 %", "50 %", "100 %"])
    ax.set_ylabel(f"semilla {seed}")
plt.show()


def stretches(days):
    # Tramos consecutivos con la misma posición objetivo.
    pos = days["position"].dropna()
    groups = (pos != pos.shift()).cumsum()
    return pd.DataFrame([
        {"desde": g.index[0].date(), "hasta": g.index[-1].date(), "días": len(g), "posición": g.iloc[0],
         "retorno del activo en el tramo": (asset_return.loc[g.index] + 1).prod() - 1}
        for _, g in pos.groupby(groups)
    ])


positions = pd.DataFrame({
    seed: {
        "días en 0 %": (d["position"] == 0).mean(),
        "días en 50 %": (d["position"] == 0.5).mean(),
        "días en 100 %": (d["position"] == 1).mean(),
        "operaciones": backtest.metrics(d)["trades"],
        "días por tramo (mediana)": stretches(d)["días"].median(),
    }
    for seed, d in agent_days.items()
}).T.rename_axis("semilla")
positions.style.format({"días en 0 %": "{:.0%}", "días en 50 %": "{:.0%}", "días en 100 %": "{:.0%}",
                        "operaciones": "{:.0f}", "días por tramo (mediana)": "{:.0f}"})

**¿Las salidas del mercado aportaron?** Para cada tramo en que una semilla
estuvo por debajo del 100 %, se compara con lo que hizo el activo durante ese
tramo. Si el activo bajó, la salida evitó una pérdida; si subió, la salida
dejó de ganar. El efecto de cada tramo frente a comprar y mantener es
aproximadamente −(1 − posición) × retorno del activo.

In [ ]:
exit_rows = []
for seed, d in agent_days.items():
    s = stretches(d)
    for _, row in s[s["posición"] < 1].iterrows():
        exit_rows.append({"semilla": seed, **row,
                          "efecto frente a comprar y mantener": -(1 - row["posición"]) * row["retorno del activo en el tramo"]})
exits = pd.DataFrame(exit_rows)
print("Efecto total de los tramos fuera del 100 %, por semilla (suma aproximada):")
print(exits.groupby("semilla")["efecto frente a comprar y mantener"].sum().map("{:+.1%}".format).to_string())
exits.sort_values("efecto frente a comprar y mantener").style.format({
    "posición": "{:.0%}", "retorno del activo en el tramo": "{:+.1%}", "efecto frente a comprar y mantener": "{:+.1%}",
}).hide(axis="index")

**Interpretación:** (se completa con los resultados)

## 28. Comparación entre las versiones de la recompensa

La recompensa v1 se eligió en la parte 9 con validación. Aquí se agrega cómo
se comportan v0 y v1 en test. v0 no es el agente final: su resultado en test
se reporta solo para completar la comparación y no cambia ninguna decisión.

In [ ]:
versions = {"v0: log-retorno": ("investor-v0", "investor-v0"),
            "v1: log-retorno − 400 bps por operar": ("investor-pen400", "investor-penalized-v0")}
version_rows = []
for name, (save, env_id) in versions.items():
    for seed in range(3):
        model = DQN.load(f"saves/dqn_{save}_seed{seed}.zip", device="cpu")
        for p in ["train", "valid", PERIOD]:
            d = backtest.run(lambda obs, m=model: m.predict(obs, deterministic=True)[0], p, env_id)
            version_rows.append({"recompensa": name, "período": p, "semilla": seed, **backtest.metrics(d)})
versions_table = (pd.DataFrame(version_rows).drop(columns=["semilla", "final_value"])
                  .groupby(["recompensa", "período"], sort=False).mean().rename(columns=LABELS))
print("Media de 3 semillas por recompensa y período")
versions_table.style.format(FORMATS)

**Interpretación:** (se completa con los resultados)

## 29. Comportamientos inesperados del agente

In [ ]:
# a) ¿Las 3 semillas coinciden? Días en que una está en 0 % y otra en 100 %.
positions_by_seed = pd.DataFrame({seed: d["position"] for seed, d in agent_days.items()}).dropna()
opposite = (positions_by_seed.max(axis=1) - positions_by_seed.min(axis=1)) == 1
print(f"Días en que las semillas toman posiciones opuestas (0 % y 100 %): {opposite.mean():.0%}")

# b) Los movimientos más grandes del período: ¿qué posición tenía cada semilla?
big = asset_return.dropna().abs().nlargest(8).index
moves = positions_by_seed.loc[big].assign(**{"retorno del activo": asset_return.loc[big]})
moves.index = [f"{d.date()} → siguiente cierre" for d in big]
moves.style.format({"retorno del activo": "{:+.1%}"} | {s: "{:.0%}" for s in agent_days})

**Comportamiento inesperado ya documentado (parte 9):** con la recompensa v0
el agente hizo *reward hacking*: memorizó el camino de precios de train
(+2.250 % en train contra +7 % en validación), y un ruido de 0,1 desviaciones
en sus observaciones le quitaba más de la mitad de su resultado en train.

**Lo que muestra el test:** (se completa con los resultados)